## Вступ   
In this chapter, we will explore a new method for testing static hypotheses     
We have already looked at the Fisher and Neyman-Pearson approaches; now we move on to     
the Bayesian approach

## What Is the Bayesian Approach ?  

It is worth recalling from Chapter 8 that:    
In predictive modelling (and statistics in general),    
the total uncertainty of our prediction of the future      
value of $Y$ based on the data $D$ is mathematically broken down into two components 

## **Law of Total Variance, EVE's law**:

$$\text{Var}(Y|D) = \underbrace{\mathbb{E}[\text{Var}(Y|\theta, D)]}_{\text{Aleatory}} + \underbrace{\text{Var}(\mathbb{E}[Y|\theta, D])}_{\text{Epistemic}}$$

$Y$ — future observation (e.g., whether the user will click).    
$D$ — historical data.   
$\theta$ — true model parameters (which we do not know).  

- **Aleatory uncertainty** (Aleatory / Statistical) — the inherent randomness of the process itself.   
Derived from the Latin word *alea* (dice). We know that the probability of a fair coin landing on heads is $0.5$,     
but we cannot predict the outcome of the next toss. In data, this is natural noise ($\epsilon$).     
**Collecting new data does NOT reduce this uncertainty.**
- **Epistemic uncertainty** (Epistemic / Systemic) — uncertainty due to our lack of knowledge of the model or its parameters.  
Derived from the Greek episteme (knowledge). For example, we do not know whether the coin is even balanced. This is   
where Bayes’ theorem works perfectly (updating the Prior to the Posterior).   
**Collecting new data REDUCES this uncertainty.**
- **Law of Total Variance** — shows that our uncertainty in a forecast consists of expected   
natural noise (Aleatory) and our uncertainty in the model itself (Epistemic).

In [4]:
import numpy as np

# Припустимо, ми моделюємо ймовірність конверсії (ctr)
# Ми маємо Posterior розподіл нашого параметра ctr після A/B тесту (Beta розподіл)
# Наприклад, a_post=150, b_post=850 (150 кліків з 1000 показів)
a_post, b_post = 150, 850

# 1. Epistemic Uncertainty (дисперсія самого параметра ctr)
# Наскільки ми не впевнені в тому, яка справжня конверсія?
# Розраховуємо дисперсію Beta-розподілу
epistemic_variance = (a_post * b_post) / ((a_post + b_post) ** 2 * (a_post + b_post + 1))


# 2. Aleatory Uncertainty (очікувана дисперсія самого результату - клік чи ні)
# Навіть якщо ми зафіксуємо ctr на його середньому значенні, 
# клік конкретного юзера - це випробування Бернуллі з дисперсією p*(1-p)
expected_ctr = a_post / (a_post + b_post)
aleatory_variance = expected_ctr * (1 - expected_ctr)

print(f"Епістемічна невизначеність (дисперсія параметрів): {epistemic_variance:.6f}")
print(f"Алеаторна невизначеність (природний шум вибірки):   {aleatory_variance:.6f}")


# Важливо: якщо ми зберемо 1 мільйон показів, epistemic_variance прямуватиме до 0,
# але aleatory_variance залишиться приблизно такою ж (близько 0.1275)

Епістемічна невизначеність (дисперсія параметрів): 0.000127
Алеаторна невизначеність (природний шум вибірки):   0.127500


### **Bayes’ fundamental theorem for data science (Hypothesis $H$ and Data $D$) is as follows:**

$$P(H|D) = \frac{P(D|H) \cdot P(H)}{P(D)}$$

$$\text{Posterior} = \frac{\text{Likelihood} \cdot \text{Prior}}{\text{Marginal Likelihood}}$$

### Example:    
Updating the beta distribution for conversion

In [6]:
import numpy as np
import scipy.stats as stats

# Prior: наші попередні уявлення про параметр ( на основі попередніх даних або інтуїції )
# a=2, b=10 означає базове очікування конверсії ~16%
alpha_prior = 2
beta_prior = 10

# Дані (Likelihood): запустили компанію, отримали 5 конверсій з 20 користувачів
successes = 5
trials = 20

# Posterior: оновлюємо знання на основі нових даних 
# Замість складних інтегралів просто додаємо успіхи до alpha, а неуспіхи - до beta
alpha_post = alpha_prior + successes
beta_post = beta_prior + (trials - successes)

# Оновлення значення
posterior_mean = alpha_post / (alpha_post + beta_post)
print(f"Оновлена ймовірність:: {posterior_mean:.3f}")

Оновлена ймовірність:: 0.219


### Задача:  
Це класична задача, відома як варіація "Парадоксу коробок Бертрана".  
У нас є 3 монети:  
- Двостороння з "Орлами" (ОО) — Heads/Heads  
- Чесна монета (ОР) — Heads/Tails  
- Двостороння з "Решками" (РР) — Tails/Tails  

Ми дістали випадкову монету, підкинули, бачимо "Орел" (Head).   
Треба знайти ймовірність того, що зі зворотного боку теж "Орел"   
(тобто, що ми витягнули монету ОО).  

Нехай подія $D$ — ми побачили "Орла" після підкидання.   
Наші гіпотези (Prior):
- $H_1$: Ми обрали монету (ОО). $P(H_1) = 1/3$   
- $H_2$: Ми обрали монету (ОР). $P(H_2) = 1/3$ 
- $H_3$: Ми обрали монету (РР). $P(H_3) = 1/3$  


Правдоподібність (Likelihood) побачити "Орла" для кожної монети:  
$P(D|H_1) = 1$ (бо там два орли)  
$P(D|H_2) = 0.5$ (бо там один орел)   
$P(D|H_3) = 0$ (бо там нуль орлів)  
Застосовуємо теорему Баєса, щоб знайти Posterior для монети (ОО):

$$P(H_1|D) = \frac{P(D|H_1) \cdot P(H_1)}{P(D|H_1)P(H_1) + P(D|H_2)P(H_2) + P(D|H_3)P(H_3)}$$

$$P(H_1|D) = \frac{1 \cdot \frac{1}{3}}{1 \cdot \frac{1}{3} + 0.5 \cdot \frac{1}{3} + 0 \cdot \frac{1}{3}} = \frac{\frac{1}{3}}{\frac{1}{3} + \frac{1}{6}} = \frac{\frac{1}{3}}{\frac{3}{6}} = \frac{2}{3}$$

### Симуляція методом Монте-Карло  

In [1]:
import numpy as np

np.random.seed(13)
n_trials = 100000

# 1: ОО, 2: ОР, 3: РР
coins = np.random.choice([1, 2, 3], size=n_trials)

# Результати підкидання (1 - Орел, 0 - Решка)
flips = np.where(
    coins == 1, 1,  # ОО завжди дає 1 (Орел)
    np.where(coins == 2, np.random.choice([0, 1], size=n_trials), # ОР дає 0 або 1
             0 # РР завжди дає 0 (Решка)
    )
)

valid_trials = coins[flips == 1]  # Вибираємо лише ті, де випав Орел

prob_OO_given_heads = np.mean(valid_trials == 1)  # Ймовірність, що монета була ОО, якщо випав Орел

print(f"Імперична ймовірність, що монета була ОО при випавшому Орлі: {prob_OO_given_heads:.3f}")


Імперична ймовірність, що монета була ОО при випавшому Орлі: 0.663


Пояснення дерева ( Expected frequency tree )  — автор книги пропонує думати не монетами, а сторонами монет (їх 6).      
З цих 6 сторін є три "Орли". Якщо ми бачимо "Орла", ми дивимось на одну з цих трьох конкретних сторін.    
Дві з цих сторін належать монеті (ОО), і лише одна — чесній монеті. Отже, шанс,    
що ми тримаємо монету (ОО) дорівнює $2$ з $3$.

![tree](tree.png)

У реальному житті спочатку йде дія, потім наслідом, автор наводить ось таку задачу:

Suppose a screening test for doping in sports is claimed to be ‘95% accurate’, meaning that 95% of dopers, and 95% of non-dopers, will be
correctly classified. Assume 1 in 50 athletes are truly doping at any time. If an athlete tests positive, what is the probability that they are truly
doping?

Але ми спостерігаємо світ з кінця: спочатку бачимо тест, і хочемо  дізнатися причину ( допінг ).

Ось як виглядає пряма ймовірність (Causation - як працює світ):  
$P(\text{Positive Test} | \text{Doping}) = 0.95$  
Ось як виглядає обернена ймовірність (Inference - як ми дізнаємося про світ):  
$P(\text{Doping} | \text{Positive Test}) = 0.28$  
Теорема Баєса — це міст між ними. Вона буквально є інструментом для обчислення "оберненої ймовірності".

## Odds and Likelihood Ratios


Ми всі чули 1 до 3 : 1/3, тобто шанс це відношення успіху до не успіху,   
приклад шанс що випаде 6 на кубику: 1/6 поділити 5/6 = 1/5 ( є одна шистірка   
а решта від 1 до 5 цифри. )

**Likelihood ratio** - це ймовірність доказів що припускають   
гіпотезу А, поділена на ймовірність доказів що припускають гіпотезу В.  
**Приклад:** 
гіпотеза А, атлет винен у допінгу;  гіпотеза В, він не винний у допінгу   
На основі поперідньої задачі: ми припускаємо  
що 95% доперів які пройдуть тест, будуть мати позитивний результат, тож  
ймовірність доказів щодо гіпотези А = 0.95  
5 % не доперів мають позитивний результа ( помилка тесту ), тож   
ймовірність доказів щодо гіпотези В = 0.05  
Тож likelihood ratio: 0.95 / 0.05 =  19, поки що ми нічого не може   
судити що це число нам показує, пізніше ми навчимося його інтерпретувати  


Тож Bayes' theorem каже:    
the final odds for the hypothesis = the initial odds for a hypothesis * the likelihood ratio  
Приклад з допінгом - на основі задачі, за умовою the initial odds = 1/49, the likelihood ratio ми порахували = 19, тоді:  
the final odds = 19 * 1/49 = 19/49   
А шанси можуть бути перетворені у ймовірність, а саме   
( заг. формула: А - позитив ісход, В - негатив ісход, тоді odds: A/B, ймовірність: A/(A+B)   
Тож: 19/(19 + 49)
The initial odds are known as prior odds;  
the final odds are known as posterior odds.  
І да це можна робити і далі і далі у глиб, по суті як рекурсія


## Likelihood Ratios and Forensic Science


Задача про:   
On Saturday 25 August 2012, archaeologists began an excavation for Richard III’s   
remains by digging in a car park in Leicester. Within a few  
hours they found their first skeleton. What is the probability that this was Richard III?  

Автор показав приклад наглядного застосування odds, Likelihood Ratio.  
Цей підхід є стандартом у криміналістиці, судах ( наприклад при аналізі  
ДНК ) - він дозволяє комбінувати багато незалежних доказів.   

$$\text{Posterior Odds} = \text{Prior Odds} \times \text{Likelihood Ratio}$$



The researchers’ own ‘sceptical’ analysis led them to posterior odds of 167,000 to 1,  
or a 0.999994 probability that they had found Richard III. This was considered sufficient   
evidence to justify burying the skeleton with full honours in Leicester Cathedral.


$$\text{Posterior Odds} \approx \frac{1}{400} \times 6,700,000 = 16,750$$


## How can we better analyse pre-election polls?


Автор показує cкладні приклади сучасного застосування цих методів   

Дві теми:   

- **MRP (Multi-level regression and post-stratification )** - Баєсівські ієрархічні моделі,    
які дозволяють роботи точні прогнози на основі дешевих, нерепрезентативних інтернет опитувань  
- **Bayes Factor** - альтернатива класичним p-values, для перевірки гіпотез  


MRP використовує часткове об'єднання ( Partial Pooling ), тобто математично припускаємо,  
що справжня ймовірність голосування в регіоні j:

$$\theta_j \sim \mathcal{N}(\mu, \sigma^2)$$


А дані опитування в цьому регіоні $y_j$ залежать від цієї   
$\theta_j$:$$y_j \sim \text{Binomial}(n_j, \theta_j)$$


Фактор Баєса ($BF$) для порівняння двох гіпотез розраховується як відношення   
маргінальних правдоподібностей:   
$$BF_{10} = \frac{P(D|H_1)}{P(D|H_0)} = \frac{\int P(D|\theta_1, H_1)P(\theta_1|H_1)d\theta_1}{\int P(D|\theta_0, H_0)P(\theta_0|H_0)d\theta_0}$$


## An Ideological Battle


Автор описує фундаментальний конфлікт про те що взагалі означає слово "ймовірність" ?  
Найбільший практичний момент - різниця між довірчими та  кредитними інтервалами.  
### **Приклад:**
Нехай ми провели А/B тест, отримали 55 конверсій із 100 показів ( 55 % ).  
Які ці дві школи опишуть невизначеність цього результату



In [2]:
import numpy as np
import scipy.stats as stats

successes, n = 55, 100
p_hat = successes / n

# Friquentist approach: (95% confidence interval for a proportion)
# Принцип: істинна конверсія це константа, вона не має ймовірності
# Випадковим є сам інтервал, який стрибає при кожній новій вибірці

se = np.sqrt(p_hat * (1 - p_hat) / n)
ci_freq = stats.norm.interval(0.95, loc=p_hat, scale=se)


# Bayesian approach: (95% credible/uncertainty interval (prior: Beta(1,1))
# Принцип: інтервал зафіксований нашими зібраними даними
# Істинна конверсія це змінна яка відобрає рівень нашого знання

alpha_post = 1 + successes
beta_post = 1 + (n - successes)
ci_bayes = (stats.beta.ppf(0.025, alpha_post, beta_post), 
            stats.beta.ppf(0.975, alpha_post, beta_post))


print("Обидва інтервали можуть бути схожими, але їх інтерпретація різна:")
print(f"Frequentist Confidence Interval: [{ci_freq[0]:.3f}, {ci_freq[1]:.3f}]")
print(f"Bayesian Credible Interval:      [{ci_bayes[0]:.3f}, {ci_bayes[1]:.3f}]")

Обидва інтервали можуть бути схожими, але їх інтерпретація різна:
Frequentist Confidence Interval: [0.452, 0.648]
Bayesian Credible Interval:      [0.452, 0.644]


Біль студентів і не тільки, **класичний 95% confidence interval**  
**НЕ ОЗНАЧАЄ** що "існує 95% ймовірність того, що істинне значення  
лежить у цьому інтервалі", воно означає якщо ми би провели експеремент  
багато разів що у 95%  випадків істинне значення містилося би у цьому інтервалі    
( хороша аналогія що є стовпчик - істинне значення, а ми на нього кидаємо кольцо   
наше припущення інтервал впененості, колода не рухома, а от ми можемо корегувати  
наші "кидки" )   
з тори зору цього підходу воно там або лежить - (100%) або не лежить - (0%).  
Натомість **Баєсівський 95% Uncertainty Interval** має саме такий простий,  
інтуїтивний зміст: "на основі цих даних я на 95% впевнений, що істина десь тут".  

## Summary:  
- Bayesian methods - це добуток наших попердніх знань ( Prior ) та нових фактів (   Likelihood )    
- Нові шанси = Старі шанси * Силу доказу   
- Коли Prior доводиться задавати "на око" - результати потребують  
перевірки, це головний мінус Bayesian methods  
- Філософські суперечки про природу статистику - це вторинне, якість  
даних, і наукова надійність ( експеремент ) - важливіше.
